# Data processing

In [1]:
import pandas as pd

# Columns to drop from the literature dataset
DROP = [
    "rxn_id",
    "ligand_1_cas",
    "ligand_1_smiles",
    "product_1_product_2_ddg_kcal",
]

# Load literature data
literature = pd.read_csv("../../../data/dft-descriptors/HF-Literature.tsv", sep="\t")

# Create a unique key for each catalyst conformer in the literature data
literature["catalyst_key"] = literature["ligand_1_cas"] + "_conformer-" + literature["conformer"].astype(str)

# Separate features (X), target (y), and grouping labels (labels) for literature data
X_lit, y_lit, labels = (
    literature.drop(columns=DROP, axis=1),
    literature["product_1_product_2_ddg_kcal"],
    literature["ligand_1_cas"],
)

# Load experimental data
experimental = pd.read_csv("../../../data/dft-descriptors/HF-Experimental.tsv", sep="\t")

# Create a unique key for each catalyst conformer in the experimental data
experimental["catalyst_key"] = experimental["ligand_1_cas"] + "_conformer-" + experimental["conformer"].astype(str)

# Get unique ligand CAS numbers from literature and experimental data
lit_names = literature.drop_duplicates(subset=["rxn_id"])["ligand_1_cas"]
exp_names = experimental.drop_duplicates(subset=["rxn_id"])["ligand_1_cas"]

In [2]:
from pathlib import Path

# Find all XYZ files in the data/xyz directory
xyzs = Path("data/xyz").rglob("*.xyz")

# Extract catalyst keys, conformer numbers, and ligand CAS numbers from filenames
catalyst_keys = [xyz.stem for xyz in xyzs]
conformers = [xyz.split("_")[-1] for xyz in catalyst_keys]  # Assuming filename format like CAS_conformer-N.xyz
conformer_ligands = [name.split("_")[0] for name in catalyst_keys]

# Create a DataFrame mapping ligand CAS to catalyst keys
conformer_df = pd.DataFrame({"ligand_1_cas": conformer_ligands, "catalyst_key": catalyst_keys})

# Load Xanthene data
xanthene_df = pd.read_csv("xanthenes-gen1.tsv", sep="\t")

# Calculate ligand-to-catalyst ratio
xanthene_df["ligand_catalyst_ratio"] = xanthene_df["ligand_1_eq"] / xanthene_df["catalyst_1_eq"]

# Merge Xanthene data with conformer data based on ligand CAS
xanthene_full = xanthene_df.merge(
    conformer_df,
    how="left",
    left_on="ligand_1_cas",
    right_on="ligand_1_cas",
)

In [3]:
import json

import numpy as np
from sklearn.model_selection import GroupShuffleSplit, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler

# Flag to control execution of this cell
run = True

if run:
    # Initialize LeaveOneGroupOut cross-validator
    logo = LeaveOneGroupOut()

    split = 0
    # Iterate through LOGO splits
    for train_val_idx, test_idx in logo.split(X_lit, y_lit, labels):
        # Initialize GroupShuffleSplit for train/validation split
        val_logo = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
        # Perform train/validation split, ensuring groups (ligands) are kept together
        train_idx, val_idx = next(
            val_logo.split(
                np.array(train_val_idx).reshape(-1, 1),  # Reshape needed for split function
                groups=labels.iloc[train_val_idx],  # Use ligand CAS as groups
            )
        )

        # Create subsets based on the indices
        train_subset = literature.iloc[train_val_idx].iloc[train_idx].copy()
        val_subset = literature.iloc[train_val_idx].iloc[val_idx].copy()
        test_subset = literature.iloc[test_idx].copy()

        # Define columns representing reaction conditions
        rxn_conds_columns = [
            "temperature_deg_c",
            "ligand_catalyst_ratio",
            "reagent_1_pressure_bar",
            "reagent_2_pressure_bar",
        ]

        # Initialize dictionary to store the split data
        dataset = {}

        # Scale reaction conditions for the literature data
        reaction_conditions = literature[rxn_conds_columns]
        scaler = StandardScaler()
        reaction_conditions = pd.DataFrame(scaler.fit_transform(reaction_conditions), columns=rxn_conds_columns)

        # Scale reaction conditions for the xanthene data using the same scaler
        xanthene_reaction_conditions = xanthene_full[rxn_conds_columns]
        xanthene_reaction_conditions = pd.DataFrame(
            scaler.transform(xanthene_reaction_conditions),
            columns=rxn_conds_columns,
        )

        # Format xanthene data for JSON output
        xanthene = {}
        xanthene["rxn_id"] = xanthene_full["rxn_id"].to_list()
        xanthene["catalyst_key"] = xanthene_full["catalyst_key"].to_list()
        xanthene["ddG"] = xanthene_full["product_1_product_2_ddg_kcal"].to_list()
        xanthene["reaction_conditions"] = xanthene_reaction_conditions.values.tolist()

        # Format training data (including xanthene data) for JSON output
        train = {}
        train["rxn_id"] = train_subset["rxn_id"].to_list() + xanthene["rxn_id"]
        train["catalyst_key"] = train_subset["catalyst_key"].to_list() + xanthene["catalyst_key"]
        train["ddG"] = train_subset["product_1_product_2_ddg_kcal"].to_list() + xanthene["ddG"]
        train["reaction_conditions"] = (
            reaction_conditions.iloc[train_val_idx].iloc[train_idx].values.tolist() + xanthene["reaction_conditions"]
        )

        # Format validation data for JSON output
        val = {}
        val["rxn_id"] = val_subset["rxn_id"].to_list()
        val["catalyst_key"] = val_subset["catalyst_key"].to_list()
        val["ddG"] = val_subset["product_1_product_2_ddg_kcal"].to_list()
        val["reaction_conditions"] = reaction_conditions.iloc[train_val_idx].iloc[val_idx].values.tolist()

        # Format test data for JSON output
        test = {}
        test["rxn_id"] = test_subset["rxn_id"].to_list()
        test["catalyst_key"] = test_subset["catalyst_key"].to_list()
        test["ddG"] = test_subset["product_1_product_2_ddg_kcal"].to_list()
        test["reaction_conditions"] = reaction_conditions.iloc[test_idx].values.tolist()

        # Add formatted train, val, test sets to the dataset dictionary
        dataset["train"] = train
        dataset["val"] = val
        dataset["test"] = test

        # Save the dataset for the current split to a JSON file
        with open(f"data/jsons/split_{split}.json", "w") as f:
            json.dump(dataset, f, indent=2)

        split += 1

    # Format experimental data for JSON output
    experimental_dataset = {}
    experimental_dataset["rxn_id"] = experimental["rxn_id"].to_list()
    experimental_dataset["catalyst_key"] = experimental["catalyst_key"].to_list()
    experimental_dataset["ddG"] = experimental["product_1_product_2_ddg_kcal"].to_list()

    # Scale experimental reaction conditions using the same scaler fitted on literature data
    experimental_reaction_conditions = experimental[rxn_conds_columns]
    experimental_reaction_conditions = pd.DataFrame(
        scaler.transform(experimental_reaction_conditions), columns=rxn_conds_columns
    )
    experimental_dataset["reaction_conditions"] = experimental_reaction_conditions.values.tolist()

    # Save the formatted experimental dataset to a JSON file
    with open("data/jsons/experimental.json", "w") as f:
        json.dump(experimental_dataset, f, indent=2)

# Results analysis

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, root_mean_squared_error

# Lists to store aggregated true values and predictions
test_ddg = []  # True ddG values from test sets across all splits
test_rmses = []  # RMSE for each split's test set
test_predictions = []  # Mean predictions for test sets across all splits
experimental_preds = []  # Predictions for the experimental set from models of each split
experimental_ddg = []  # True ddG values for the experimental set (repeated for each split)

# Iterate through potential split directories (assuming up to 156 splits)
for split in range(156):
    try:
        split_path = Path(f"models/split_{split:03d}")
        if not split_path.exists():
            continue  # Skip if directory doesn't exist

        result_jsons = list(split_path.rglob("results-last.json"))
        if not result_jsons:
            print(f"No results.json found for split {split}")
            continue

        internal_test_ddg = []
        internal_test_rmses = []
        internal_test_predictions = []

        for result_json in result_jsons:
            metrics = result_json.parent / "metrics.csv"
            if metrics.exists():
                df = pd.read_csv(metrics)
                if not df.empty:
                    last_row = df.iloc[-1]
                    last_epoch = int(last_row["epoch"])

            args = result_json.parent / "args.json"
            if args.exists():
                with open(args) as f:
                    args_data = json.load(f)

            try:
                with open(result_json) as f:
                    dataset = json.load(f)
            except json.JSONDecodeError:
                print(f"Error decoding JSON in {result_json}")
                continue

            try:
                # Group by reaction ID and calculate mean across conformers
                seed = result_json.parents[2].name
                test = pd.DataFrame(dataset["test"]).groupby("rxn_id").mean(numeric_only=True)

            except (ValueError, KeyError) as e:
                print(f"Error processing test data in {result_json}: {e}")
                continue

            test_rmse = root_mean_squared_error(test["ddG"], test["y_pred"])
            internal_test_ddg.append(test["ddG"])
            internal_test_rmses.append(test_rmse)
            internal_test_predictions.append(test["y_pred"])

        # Aggregate results for this split (average across runs if multiple exist)
        if internal_test_ddg:
            test_ddg.extend(np.mean(internal_test_ddg, axis=0).tolist())
            test_predictions.extend(np.mean(internal_test_predictions, axis=0).tolist())  #
            test_rmses.append(np.mean(internal_test_rmses))
        else:
            print(f"No valid internal test results processed for split {split}")

        # Find and process experimental result files for this split
        exp_jsons = list(split_path.rglob("exp_results-last.json"))
        internal_exp_ddg = []
        internal_exp_preds = []
        for exp_json in exp_jsons:
            try:
                with open(exp_json) as f:
                    exp_dataset = json.load(f)
            except json.JSONDecodeError:
                print(f"Error decoding JSON in {exp_json}")
                continue

            try:
                # Group experimental results by reaction ID and average across conformers
                exp = pd.DataFrame(exp_dataset).groupby("rxn_id").mean(numeric_only=True)
            except (ValueError, KeyError) as e:
                print(f"Error processing experimental data in {exp_json}: {e}")
                continue

            # Store experimental results for averaging across runs within this split
            internal_exp_ddg.append(exp["ddG"])
            internal_exp_preds.append(exp["y_pred"])

        # Append the averaged experimental predictions and true values for this split
        if internal_exp_ddg:
            experimental_ddg.append(np.mean(internal_exp_ddg, axis=0))
            experimental_preds.append(np.mean(internal_exp_preds, axis=0))
        elif exp_jsons:
            print(f"No valid internal experimental results processed for split {split}")

    except FileNotFoundError:
        print(f"Split directory models/split_{split:03d} not found")
    except Exception as e:
        print(f"An unexpected error occurred processing split {split}: {e}")

# Calculate overall LOCO metrics for the aggregated test set predictions
if test_ddg and test_predictions:
    loco_rmse = root_mean_squared_error(test_ddg, test_predictions)
    loco_r2 = r2_score(test_ddg, test_predictions)
    print(f"Total aggregated test predictions: {len(test_ddg)}")
    print(f"LOCO Test RMSE: {loco_rmse:.2f}, R2: {loco_r2:.2f}")
else:
    print("No test predictions aggregated.")
    loco_rmse, loco_r2 = np.nan, np.nan

# Calculate overall metrics for the aggregated experimental set predictions
# Average the predictions across all splits for the final experimental prediction
if experimental_ddg and experimental_preds:
    final_experimental_ddg = experimental_ddg[0].tolist()
    final_experimental_preds = np.mean(experimental_preds, axis=0).tolist()

    print(f"Total aggregated experimental predictions: {len(final_experimental_ddg)}")

    exp_r2 = r2_score(final_experimental_ddg, final_experimental_preds)
    exp_rmse = root_mean_squared_error(final_experimental_ddg, final_experimental_preds)
    print(f"Experimental RMSE: {exp_rmse:.2f}, R2: {exp_r2:.2f}")
else:
    print("No experimental predictions aggregated.")
    final_experimental_ddg, final_experimental_preds = [], []
    exp_r2, exp_rmse = np.nan, np.nan

Total aggregated test predictions: 378
LOCO Test RMSE: 0.65, R2: 0.74
Total aggregated experimental predictions: 41
Experimental RMSE: 0.46, R2: 0.78
